<a href="https://colab.research.google.com/github/lekhasree15/AI_LAB/blob/main/lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
class KnowledgeBase:
    def __init__(self):
        # Facts represent simple ground predicates: (predicate_name, tuple_of_arguments)
        self.facts = set()
        # Rules are tuples: (list_of_conditions, conclusion_predicate, mapping_function_or_variables)
        self.rules = []

    def add_fact(self, fact):
        self.facts.add(fact)

    def add_rule(self, premises, conclusion):
        # premises: list of (pred_name, vars)
        # conclusion: (pred_name, vars)
        self.rules.append((premises, conclusion))

    def forward_chain(self, max_iterations=10):
        import itertools

        print("Initial Facts:")
        for f in sorted(self.facts):
            print(f"  {f[0]}({', '.join(f[1])})")
        print("\n--- Starting Forward Reasoning ---")

        new_facts_added = True
        iteration = 0

        # Collect all constants currently in our domain to bind to variables
        constants = set()
        for fact in self.facts:
            constants.update(fact[1])

        while new_facts_added and iteration < max_iterations:
            new_facts_added = False
            iteration += 1

            # We look at each rule and try all possible variable assignments from our constants pool
            for premises, conclusion in self.rules:
                # Collect all unique variables in this rule
                variables = set()
                for pred, args in premises:
                    for arg in args:
                        if arg.islower(): # simple convention: lowercase strings are variables
                            variables.add(arg)

                var_list = list(variables)
                if not var_list:
                    # Ground rule (no variables)
                    if all((pred, args) in self.facts for pred, args in premises):
                        if conclusion not in self.facts:
                            self.facts.add(conclusion)
                            constants.update(conclusion[1])
                            print(f"Derived: {conclusion[0]}({', '.join(conclusion[1])}) using ground rule.")
                            new_facts_added = True
                    continue

                # Try all combinations of constants bound to variables
                for combination in itertools.product(constants, repeat=len(var_list)):
                    binding = dict(zip(var_list, combination))

                    # Helper to substitute variables with bound constants
                    def subst(args):
                        return tuple(binding.get(arg, arg) for arg in args)

                    # Check if all premises under this substitution exist as facts
                    satisfied = True
                    for pred, args in premises:
                        substituted_args = subst(args)
                        if (pred, substituted_args) not in self.facts:
                            satisfied = False
                            break

                    if satisfied:
                        sub_conclusion = (conclusion[0], subst(conclusion[1]))
                        if sub_conclusion not in self.facts:
                            self.facts.add(sub_conclusion)
                            constants.update(sub_conclusion[1])
                            explanation = ", ".join([f"{p}({', '.join(subst(a))})" for p, a in premises])
                            print(f"Derived: {sub_conclusion[0]}({', '.join(sub_conclusion[1])}) from [{explanation}]")
                            new_facts_added = True

        print("\n--- Forward Reasoning Completed ---")

# Initialize Knowledge Base
kb = KnowledgeBase()

# Add initial facts
kb.add_fact(("Enemy", ("A", "America")))
kb.add_fact(("American", ("Robert",)))
kb.add_fact(("Missile", ("M1",)))
kb.add_fact(("Owns", ("A", "M1")))

# Add Rules
# 1. Enemy(x, America) => Hostile(x)
kb.add_rule([("Enemy", ("x", "America"))], ("Hostile", ("x",)))

# 2. Missile(x) => Weapon(x)
kb.add_rule([("Missile", ("x",))], ("Weapon", ("x",)))

# 3. Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)
kb.add_rule([("Missile", ("x",)), ("Owns", ("A", "x"))], ("Sells", ("Robert", "x", "A")))

# 4. American(x) ^ Weapon(y) ^ Hostile(z) ^ Sells(x, y, z) => Criminal(x)
kb.add_rule([
    ("American", ("x",)),
    ("Weapon", ("y",)),
    ("Hostile", ("z",)),
    ("Sells", ("x", "y", "z"))
], ("Criminal", ("x",)))

# Run the proof
kb.forward_chain()

# Verify Query
is_proven = ("Criminal", ("Robert",)) in kb.facts
print(f"\nQuery 'Criminal(Robert)' is PROVEN: {is_proven}")

Initial Facts:
  American(Robert)
  Enemy(A, America)
  Missile(M1)
  Owns(A, M1)

--- Starting Forward Reasoning ---
Derived: Hostile(A) from [Enemy(A, America)]
Derived: Weapon(M1) from [Missile(M1)]
Derived: Sells(Robert, M1, A) from [Missile(M1), Owns(A, M1)]
Derived: Criminal(Robert) from [American(Robert), Weapon(M1), Hostile(A), Sells(Robert, M1, A)]

--- Forward Reasoning Completed ---

Query 'Criminal(Robert)' is PROVEN: True
